# Notebook 03: Model Training
**Thesis:** Autonomous Threat Hunting: ML-Based MITRE ATT&CK Technique Detection

In [15]:
import pandas as pd
import numpy as np
import os, pickle, warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

print('=== NOTEBOOK 03: MODEL TRAINING ===')
print('Libraries imported successfully')

=== NOTEBOOK 03: MODEL TRAINING ===
Libraries imported successfully


In [16]:
# Load data
data_dir = '../data/processed'
X_train   = pd.read_csv(f'{data_dir}/X_train.csv')
X_test    = pd.read_csv(f'{data_dir}/X_test.csv')
y_train   = pd.read_csv(f'{data_dir}/y_train.csv').squeeze()
y_test    = pd.read_csv(f'{data_dir}/y_test.csv').squeeze()
label_map = pd.read_csv(f'{data_dir}/label_map.csv')

# Encode any remaining text columns
text_cols = X_train.select_dtypes(include='object').columns.tolist()
for col in text_cols:
    le = LabelEncoder()
    all_vals = pd.concat([X_train[col], X_test[col]]).astype(str)
    le.fit(all_vals)
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col]  = le.transform(X_test[col].astype(str))

X_train = X_train.fillna(0)
X_test  = X_test.fillna(0)

# Build target_names from ACTUAL classes in y_test (not label_map which has 3 entries)
actual_classes = sorted(y_test.unique())
target_names   = [label_map.loc[label_map['encoded'] == i, 'technique'].values[0] for i in actual_classes]

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'Classes in data: {target_names}')
print('Data ready for training!')

X_train: (9160, 14), X_test: (2291, 14)
Classes in data: ['T1021', 'T1110']
Data ready for training!


In [17]:
# MODEL 1: RANDOM FOREST
print('=== MODEL 1: RANDOM FOREST ===')
print('Training...')

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

rf_acc = accuracy_score(y_test, rf_pred)
rf_pre = precision_score(y_test, rf_pred, average='weighted')
rf_rec = recall_score(y_test, rf_pred, average='weighted')
rf_f1  = f1_score(y_test, rf_pred, average='weighted')

print(f'Accuracy:  {rf_acc:.4f} ({rf_acc*100:.2f}%)')
print(f'Precision: {rf_pre:.4f}')
print(f'Recall:    {rf_rec:.4f}')
print(f'F1 Score:  {rf_f1:.4f}')
print()
print(classification_report(y_test, rf_pred, labels=actual_classes, target_names=target_names))

=== MODEL 1: RANDOM FOREST ===
Training...
Accuracy:  0.9782 (97.82%)
Precision: 0.9787
Recall:    0.9782
F1 Score:  0.9752

              precision    recall  f1-score   support

       T1021       0.98      1.00      0.99      2178
       T1110       1.00      0.56      0.72       113

    accuracy                           0.98      2291
   macro avg       0.99      0.78      0.85      2291
weighted avg       0.98      0.98      0.98      2291



In [18]:
# MODEL 2: XGBOOST
print('=== MODEL 2: XGBOOST ===')
print('Training...')

xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6,
                           random_state=42, eval_metric='mlogloss', verbosity=0)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_pre = precision_score(y_test, xgb_pred, average='weighted')
xgb_rec = recall_score(y_test, xgb_pred, average='weighted')
xgb_f1  = f1_score(y_test, xgb_pred, average='weighted')

print(f'Accuracy:  {xgb_acc:.4f} ({xgb_acc*100:.2f}%)')
print(f'Precision: {xgb_pre:.4f}')
print(f'Recall:    {xgb_rec:.4f}')
print(f'F1 Score:  {xgb_f1:.4f}')
print()
print(classification_report(y_test, xgb_pred, labels=actual_classes, target_names=target_names))

=== MODEL 2: XGBOOST ===
Training...
Accuracy:  0.9782 (97.82%)
Precision: 0.9787
Recall:    0.9782
F1 Score:  0.9752

              precision    recall  f1-score   support

       T1021       0.98      1.00      0.99      2178
       T1110       1.00      0.56      0.72       113

    accuracy                           0.98      2291
   macro avg       0.99      0.78      0.85      2291
weighted avg       0.98      0.98      0.98      2291



In [19]:
# MODEL 3: NEURAL NETWORK (MLP)
print('=== MODEL 3: NEURAL NETWORK (MLP) ===')
print('Training...')

mlp_model = MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu',
                           max_iter=300, random_state=42,
                           early_stopping=True, validation_fraction=0.1)
mlp_model.fit(X_train, y_train)
mlp_pred = mlp_model.predict(X_test)

mlp_acc = accuracy_score(y_test, mlp_pred)
mlp_pre = precision_score(y_test, mlp_pred, average='weighted')
mlp_rec = recall_score(y_test, mlp_pred, average='weighted')
mlp_f1  = f1_score(y_test, mlp_pred, average='weighted')

print(f'Accuracy:  {mlp_acc:.4f} ({mlp_acc*100:.2f}%)')
print(f'Precision: {mlp_pre:.4f}')
print(f'Recall:    {mlp_rec:.4f}')
print(f'F1 Score:  {mlp_f1:.4f}')
print(f'(Stopped after {mlp_model.n_iter_} iterations)')
print()
print(classification_report(y_test, mlp_pred, labels=actual_classes, target_names=target_names))

=== MODEL 3: NEURAL NETWORK (MLP) ===
Training...
Accuracy:  0.9625 (96.25%)
Precision: 0.9575
Recall:    0.9625
F1 Score:  0.9547
(Stopped after 36 iterations)

              precision    recall  f1-score   support

       T1021       0.97      1.00      0.98      2178
       T1110       0.80      0.32      0.46       113

    accuracy                           0.96      2291
   macro avg       0.88      0.66      0.72      2291
weighted avg       0.96      0.96      0.95      2291



In [20]:
# COMPARE ALL 3 MODELS
print('=== MODEL COMPARISON ===')
print()

results = pd.DataFrame({
    'Model':     ['Random Forest', 'XGBoost', 'Neural Network (MLP)'],
    'Accuracy':  [rf_acc,  xgb_acc,  mlp_acc],
    'Precision': [rf_pre,  xgb_pre,  mlp_pre],
    'Recall':    [rf_rec,  xgb_rec,  mlp_rec],
    'F1 Score':  [rf_f1,   xgb_f1,   mlp_f1]
})

display_df = results.copy()
for col in ['Accuracy','Precision','Recall','F1 Score']:
    display_df[col] = display_df[col].apply(lambda x: f'{x:.4f}')
print(display_df.to_string(index=False))
print()

best_idx = results['F1 Score'].idxmax()
print(f'Best model: {results.loc[best_idx, "Model"]} (F1={results.loc[best_idx, "F1 Score"]:.4f})')

=== MODEL COMPARISON ===

               Model Accuracy Precision Recall F1 Score
       Random Forest   0.9782    0.9787 0.9782   0.9752
             XGBoost   0.9782    0.9787 0.9782   0.9752
Neural Network (MLP)   0.9625    0.9575 0.9625   0.9547

Best model: Random Forest (F1=0.9752)


In [21]:
# SAVE EVERYTHING
print('=== SAVING MODELS AND RESULTS ===')

os.makedirs('../models', exist_ok=True)
os.makedirs('../results', exist_ok=True)

for name, model in [('random_forest', rf_model), ('xgboost', xgb_model), ('neural_network', mlp_model)]:
    with open(f'../models/{name}.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f'  Saved: models/{name}.pkl')

results.to_csv('../results/model_comparison.csv', index=False)
print('  Saved: results/model_comparison.csv')

pd.DataFrame({'y_true': y_test.values, 'rf_pred': rf_pred,
              'xgb_pred': xgb_pred, 'mlp_pred': mlp_pred
}).to_csv('../results/predictions.csv', index=False)
print('  Saved: results/predictions.csv')

pd.DataFrame({
    'feature': X_train.columns,
    'importance_rf':  rf_model.feature_importances_,
    'importance_xgb': xgb_model.feature_importances_
}).sort_values('importance_rf', ascending=False).to_csv('../results/feature_importance.csv', index=False)
print('  Saved: results/feature_importance.csv')

print()

=== SAVING MODELS AND RESULTS ===
  Saved: models/random_forest.pkl
  Saved: models/xgboost.pkl
  Saved: models/neural_network.pkl
  Saved: results/model_comparison.csv
  Saved: results/predictions.csv
  Saved: results/feature_importance.csv

